# 04 — Apply HMRF Spatial Regularization

Refine GMM labels using a Hidden Markov Random Field with Potts prior. Neighboring voxels are encouraged to share the same label.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from research_ct.io.volume_saver import Load_From_Numpy
from research_ct.segmentation.hmrf import Hmrf_Segmenter

# Load GMM outputs
Processed = Load_From_Numpy("../data/processed/preprocessed_volume.npz")
Labels_Gmm = Load_From_Numpy("../data/output/gmm_labels.npz")
Probs_Gmm = Load_From_Numpy("../data/output/gmm_probabilities.npz")

print(f"Processed: {Processed.shape}")
print(f"Labels: {Labels_Gmm.shape}")
print(f"Probabilities: {Probs_Gmm.shape}")

## Compute Log-Probabilities

HMRF needs log p(x_i | k) for the energy function.

In [ ]:
# Clip to avoid log(0)
Log_Probs = np.log(np.clip(Probs_Gmm, 1e-10, 1.0))
print(f"Log-probabilities range: [{Log_Probs.min():.2f}, {Log_Probs.max():.2f}]")

## Run HMRF

ICM optimization with Potts prior. Beta controls spatial smoothness strength.

- Low beta (~0.1): weak smoothing, preserves fine details
- High beta (~2.0): strong smoothing, removes noise

Start with beta=0.5 and adjust based on results.

In [ ]:
# For large volumes, run on a subset first to tune beta
Test_Z_Range = slice(0, min(50, Processed.shape[0]))

Hmrf = Hmrf_Segmenter(
    Beta=0.5,
    Max_Iterations=20,
    Connectivity=6,
)

Labels_Hmrf_Test = Hmrf.Fit(
    Processed[Test_Z_Range],
    Log_Probs[Test_Z_Range],
)

## Compare GMM vs HMRF (Test Region)

Visualize the effect of spatial regularization.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 15))

for i, z in enumerate([10, 25, 40]):
    # Original
    axes[0, i].imshow(Processed[z], cmap='gray')
    axes[0, i].set_title(f'Processed — Z={z}')
    axes[0, i].axis('off')
    
    # GMM only
    axes[1, i].imshow(Labels_Gmm[z], cmap='tab10', vmin=0, vmax=Labels_Gmm.max())
    axes[1, i].set_title(f'GMM — Z={z}')
    axes[1, i].axis('off')
    
    # HMRF
    axes[2, i].imshow(Labels_Hmrf_Test[z], cmap='tab10', vmin=0, vmax=Labels_Gmm.max())
    axes[2, i].set_title(f'HMRF — Z={z}')
    axes[2, i].axis('off')

axes[0, 0].set_ylabel('Intensity', fontsize=12)
axes[1, 0].set_ylabel('GMM Only', fontsize=12)
axes[2, 0].set_ylabel('HMRF', fontsize=12)
plt.suptitle('Spatial Regularization Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## Run on Full Volume (if satisfied with beta)

Once beta is tuned, apply to the entire volume. This may take 30+ minutes.

In [ ]:
# Uncomment to run on full volume
# Hmrf_Full = Hmrf_Segmenter(Beta=0.5, Max_Iterations=50, Connectivity=6)
# Labels_Hmrf_Full = Hmrf_Full.Fit(Processed, Log_Probs)
# 
# Save_As_Numpy(Labels_Hmrf_Full.astype(np.uint8), "../data/output/hmrf_labels.npz")

print("Uncomment the cell above to run HMRF on full volume")

## Quantitative Comparison

Compare label statistics before and after HMRF.

In [ ]:
from research_ct.analysis.material_stats import Compute_Material_Statistics

# GMM stats
Stats_Gmm = Compute_Material_Statistics(Processed, Labels_Gmm, Num_Classes=int(Labels_Gmm.max()+1))

print("GMM Segmentation:")
print(f"  Classes found: {len(Stats_Gmm['classes'])}")
for c in Stats_Gmm['classes']:
    print(f"  Class {c['class_id']}: {c['voxel_count']:,} voxels ({c['volume_fraction']:.2%})")

# If HMRF full run completed, compare
# Stats_Hmrf = Compute_Material_Statistics(Processed, Labels_Hmrf_Full, ...)
# print("\nHMRF Segmentation:")
# ...